In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf  # For candlestick charts
import logging
import json 
import numpy as np

def process_tick_data():
    """Process collected tick data and compute stock movements."""
    logging.info("Processing Tick Data...")

    with open('/Users/mahesh/Documents/DE_learning/fundsandsavings/tick_stock_data_2025-03-13.txt', "r") as file:
        raw_text = file.read()

    data = []
    for line in raw_text.strip().split('\n'):
        try:
            timestamp, dict_str = line.split(", ", 1)
            dict_data = json.loads(dict_str)
            dict_data['timestamp'] = timestamp

            dict_data["h"] = dict_data.get("h", dict_data["lp"])
            dict_data["l"] = dict_data.get("l", dict_data["lp"])
            dict_data["ap"] = dict_data.get("ap", dict_data["lp"])

            data.append(dict_data)

        except Exception as e:
            logging.warning(f"Skipping malformed line: {line} | Error: {e}")

    df = pd.DataFrame(data)

    all_columns = ["timestamp", "t", "e", "tk", "lp", "pc", "ft", "h", "l", "ap", "v", "bp1", "sp1", "bq1", "sq1"]
    df = df.reindex(columns=all_columns, fill_value="")
    
    numeric_columns = ["lp", "pc", "ft", "h", "l", "ap", "v", "bp1", "sp1", "bq1", "sq1"]
    df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

    if df.empty:
        logging.error("No tick data available for processing.")
        return None

    df['timestamp'] = pd.to_datetime(df['timestamp']) 
    df.set_index('timestamp', inplace=True)

    
    return df

def atr(DF,n):
    "function to calculate True Range and Average True Range"
    df = DF.copy()
    df['H-L']=abs(df['high']-df['low'])
    df['H-PC']=abs(df['high']-df['close'].shift(1))
    df['L-PC']=abs(df['low']-df['close'].shift(1))
    df['TR']=df[['H-L','H-PC','L-PC']].max(axis=1,skipna=False)
    df['ATR'] = df['TR'].ewm(com=n,min_periods=n).mean()
    return df['ATR']

def supertrend(DF,n,m):
    """function to calculate Supertrend given historical candle data
        n = n day ATR - usually 7 day ATR is used
        m = multiplier - usually 2 or 3 is used"""
    df = DF.copy()
    df['ATR'] = atr(df,n)
    df["B-U"]=((df['high']+df['low'])/2) + m*df['ATR'] 
    df["B-L"]=((df['high']+df['low'])/2) - m*df['ATR']
    df["U-B"]=df["B-U"]
    df["L-B"]=df["B-L"]

    ind = df.index
    for i in range(n,len(df)):
        if df['close'][i-1]<=df['U-B'][i-1]:
            df.loc[ind[i],'U-B']=min(df['B-U'][i],df['U-B'][i-1])
        else:
            df.loc[ind[i],'U-B']=df['B-U'][i]    
    for i in range(n,len(df)):
        if df['close'][i-1]>=df['L-B'][i-1]:
            df.loc[ind[i],'L-B']=max(df['B-L'][i],df['L-B'][i-1])
        else:
            df.loc[ind[i],'L-B']=df['B-L'][i]  
    df['Strend']=np.nan
    for test in range(n,len(df)):
        if df['close'][test-1]<=df['U-B'][test-1] and df['close'][test]>df['U-B'][test]:
            df.loc[ind[test],'Strend']=df['L-B'][test]
            break
        if df['close'][test-1]>=df['L-B'][test-1] and df['close'][test]<df['L-B'][test]:
            df.loc[ind[test],'Strend']=df['U-B'][test]
            break
    
    for i in range(n,len(df)):
        
        if df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]<=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
        elif  df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]>=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]>=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]<=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
    return df[['Strend']] 

# Apply Supertrend calculation

# while(1):

for i in ['18921']:
        df = process_tick_data() 
        df = df[df['tk'] == i]

        ohlc_df = df.groupby("tk").resample("5T").agg({
        "lp": ["first", "last"],  # Open and Close prices
        "h": "max",  # Highest price in interval
        "l": "min",  # Lowest price in interval
        "v": "sum"  # Total volume traded
    }).dropna()
        ohlc_df.columns = ['open','close',  'high', 'low',   'price_change']
        
        ohlc_df = ohlc_df.reset_index() 

        ohlc_df['supertrend_value'] = supertrend(  ohlc_df,n=1, m=1 )
        ohlc_df['supertrend_signal'] = ohlc_df.apply(lambda row: "CLOSE" if row['close'] > row['supertrend_value'] else "OPEN", axis=1)
    
        print( ohlc_df )
        last_row = ohlc_df.iloc[-1]
        last_supertrend_signal = last_row['supertrend_signal']

        print(f"[{i}] - Supertrend Signal:", last_supertrend_signal)

        # if last_supertrend_signal == 'CLOSE':
        #     print(f"[{i}] EXIT - Supertrend Signal:", last_supertrend_signal)
        #     exit()



      tk           timestamp    open   close    high     low  price_change   
0  18921 2025-03-13 09:15:00  485.35  484.25  485.75  483.60    14473517.0  \
1  18921 2025-03-13 09:20:00  484.20  484.30  484.35  484.20      435887.0   
2  18921 2025-03-13 11:40:00  493.25  493.40  493.40  493.15    26719953.0   
3  18921 2025-03-13 12:25:00  491.90  490.20  491.90  490.20    80322992.0   
4  18921 2025-03-13 12:30:00  490.20  489.95  490.20  489.95    10229917.0   
5  18921 2025-03-13 12:35:00  489.75  490.40  490.40  489.75           0.0   
6  18921 2025-03-13 12:40:00  490.40  490.40  490.40  490.40     5189284.0   
7  18921 2025-03-13 12:45:00  491.20  491.60  491.75  491.00    10593034.0   
8  18921 2025-03-13 12:55:00  491.40  491.40  491.40  491.40     5384837.0   

   supertrend_value supertrend_signal  
0               NaN              OPEN  
1               NaN              OPEN  
2        487.158333             CLOSE  
3        487.158333             CLOSE  
4        487.865000

In [52]:

from algo_lib import * 

message = f"""✅ 
SELL TRADE
SYMBOL : trading_symbol
PRICE latest_price
transtype = transtype 
exch = exch
symbol_id = symbol_id
quantity = quantity
max lot = max_lots """

send_telegram_alert(message)
                                

Alert sent successfully!


In [32]:
import pandas as pd
import numpy as np
import json
import logging

def atr(DF, n):
    """Calculate True Range (TR) and Average True Range (ATR)."""
    df = DF.copy()
    df['H-L'] = abs(df['high'] - df['low'])
    df['H-PC'] = abs(df['high'] - df['close'].shift(1))
    df['L-PC'] = abs(df['low'] - df['close'].shift(1))
    df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1, skipna=False)
    df['ATR'] = df['TR'].ewm(com=n, min_periods=n).mean()
    return df['ATR']

def supertrend(DF, n, m):
    """Calculate Supertrend on given OHLC data."""
    df = DF.copy()
    df['ATR'] = atr(df, n)
    
    df["B-U"] = ((df['high'] + df['low']) / 2) + m * df['ATR']
    df["B-L"] = ((df['high'] + df['low']) / 2) - m * df['ATR']
    df["U-B"] = df["B-U"]
    df["L-B"] = df["B-L"]
    
    ind = df.index
    for i in range(n, len(df)):
        if df['close'][i - 1] <= df['U-B'][i - 1]:
            df.loc[ind[i], 'U-B'] = min(df['B-U'][i], df['U-B'][i - 1])
        else:
            df.loc[ind[i], 'U-B'] = df['B-U'][i]    
            
    for i in range(n, len(df)):
        if df['close'][i - 1] >= df['L-B'][i - 1]:
            df.loc[ind[i], 'L-B'] = max(df['B-L'][i], df['L-B'][i - 1])
        else:
            df.loc[ind[i], 'L-B'] = df['B-L'][i]  
            
    df['Strend'] = np.nan
    
    test = n
    while test < len(df):
        if df['close'][test - 1] <= df['U-B'][test - 1] and df['close'][test] > df['U-B'][test]:
            df.loc[ind[test], 'Strend'] = df['L-B'][test]
            break
        if df['close'][test - 1] >= df['L-B'][test - 1] and df['close'][test] < df['L-B'][test]:
            df.loc[ind[test], 'Strend'] = df['U-B'][test]
            break
        test += 1

    for i in range(test + 1, len(df)):
        prev_strend = df['Strend'][i - 1]
        if prev_strend == df['U-B'][i - 1] and df['close'][i] <= df['U-B'][i]:
            df.loc[ind[i], 'Strend'] = df['U-B'][i]
        elif prev_strend == df['U-B'][i - 1] and df['close'][i] >= df['U-B'][i]:
            df.loc[ind[i], 'Strend'] = df['L-B'][i]
        elif prev_strend == df['L-B'][i - 1] and df['close'][i] >= df['L-B'][i]:
            df.loc[ind[i], 'Strend'] = df['L-B'][i]
        elif prev_strend == df['L-B'][i - 1] and df['close'][i] <= df['L-B'][i]:
            df.loc[ind[i], 'Strend'] = df['U-B'][i]

    return df['Strend']

def process_tick_data():
    """Process tick data, resample to 5-minute OHLC, and compute Supertrend."""
    logging.info("Processing Tick Data...")

    with open("/Users/mahesh/Documents/DE_learning/fundsandsavings/tick_stock_data_2025-03-11.txt", "r") as file:
        raw_text = file.read()

    data = []
    for line in raw_text.strip().split('\n'):
        try:
            timestamp, dict_str = line.split(", ", 1)
            dict_data = json.loads(dict_str)
            dict_data['timestamp'] = timestamp
            data.append(dict_data)
        except Exception as e:
            logging.warning(f"Skipping malformed line: {line} | Error: {e}")

    df = pd.DataFrame(data)

    # Ensure 'tk' column is a string before filtering
    df['tk'] = df['tk'].astype(str)  
    df = df[df['tk'] == '2031']

    # Convert timestamp to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Convert numeric columns to float
    numeric_columns = ['lp', 'pc']
    df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

    # Resample tick data to 5-minute OHLC
    df.set_index('timestamp', inplace=True)
    ohlc_df = df.resample('5T').agg({
        'lp': ['first', 'max', 'min', 'last'],
        'pc': 'sum'
    }).dropna()

    # Rename columns
    ohlc_df.columns = ['open', 'high', 'low', 'close', 'volume']
    ohlc_df.reset_index(inplace=True)

    # Compute Supertrend
    ohlc_df['Supertrend'] = supertrend(ohlc_df, n=1, m=1)

    return ohlc_df

# Run the process
df_5min = process_tick_data()
df_5min


,timestamp,open,high,low,close,volume,Supertrend
0,2025-03-11 09:15:00,2678.95,2678.95,2650.0,2661.6,-493.10,NaN
1,2025-03-11 09:20:00,2661.75,2661.75,2638.8,2646.7,-612.61,NaN
2,2025-03-11 09:25:00,2645.50,2650.00,2645.5,2646.4,-6.14,NaN


In [12]:
import pandas as pd
import numpy as np
import json
import logging

with open("/Users/mahesh/Documents/DE_learning/fundsandsavings/tick_stock_data_2025-03-12.txt", "r") as file:
        data = file.read()

# Processing the data
rows = []
for line in data.strip().split("\n"):
    timestamp, json_part = line.split(", ", 1)  # Split timestamp and JSON
    json_data = json.loads(json_part)          # Convert JSON string to dictionary
    json_data["timestamp"] = timestamp         # Add timestamp

    json_data["h"] = json_data.get("h", json_data["lp"])
    json_data["l"] = json_data.get("l", json_data["lp"])
    json_data["ap"] = json_data.get("ap", json_data["lp"])
    
    rows.append(json_data)

# Convert to DataFrame
df = pd.DataFrame(rows)

# Ensure all possible columns exist, filling missing ones with empty values
all_columns = ["timestamp", "t", "e", "tk", "lp", "pc", "ft", "h", "l", "ap", "v", "bp1", "sp1", "bq1", "sq1"]
df = df.reindex(columns=all_columns, fill_value="")

# Convert numeric fields to appropriate types
numeric_columns = ["lp", "pc", "ft", "h", "l", "ap", "v", "bp1", "sp1", "bq1", "sq1"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

df = df[df['tk'] == '8479']
# Print the DataFrame
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.set_index("timestamp", inplace=True)

# Resample to 5-minute intervals, grouping by 'tk' (ticker)
df_resampled = df.groupby("tk").resample("5T").agg({
    "lp": ["first", "last"],  # Open and Close prices
    "h": "max",  # Highest price in interval
    "l": "min",  # Lowest price in interval
    "v": "sum"  # Total volume traded
})

# Rename columns
df_resampled.columns = ["open", "close", "high", "low", "volume"]
df_resampled.reset_index(inplace=True)

df_resampled

,tk,timestamp,open,close,high,low,volume
0,8479,2025-03-12 09:15:00,2290.25,2268.50,2293.95,2268.45,566887.0
1,8479,2025-03-12 09:20:00,2269.25,2264.85,2269.30,2264.50,314082.0
2,8479,2025-03-12 09:25:00,2264.45,2258.35,2266.55,2256.00,2358274.0
3,8479,2025-03-12 09:30:00,2257.65,2262.95,2263.90,2255.10,694079.0
4,8479,2025-03-12 09:35:00,2264.40,2267.50,2269.30,2260.75,657132.0
...,...,...,...,...,...,...,...
61,8479,2025-03-12 14:20:00,2258.00,2258.25,2260.20,2256.00,12263805.0
62,8479,2025-03-12 14:25:00,2257.75,2257.55,2261.00,2256.95,14398791.0
63,8479,2025-03-12 14:30:00,2257.35,2259.65,2261.00,2256.10,16346838.0
64,8479,2025-03-12 14:35:00,2260.85,2261.80,2262.50,2259.50,14705028.0


In [14]:
import pandas as pd

def calculate_supertrend_5min(df, length=1, factor=1):
    
    def compute_supertrend(group):
        # Ensure sorting by time
        group = group.sort_values(by='timestamp').reset_index(drop=True)

        print(len(group) ) 
        # Calculate True Range (TR)
        group['previous_close'] = group['close'].shift(1)
        group['tr1'] = group['high'] - group['low']
        group['tr2'] = abs(group['high'] - group['previous_close'])
        group['tr3'] = abs(group['low'] - group['previous_close'])
        group['tr'] = group[['tr1', 'tr2', 'tr3']].max(axis=1)

        # Compute ATR (Using Exponential Moving Average for better responsiveness)
        group['atr'] = group['tr'].ewm(span=length, adjust=False).mean()

        # Calculate HL2 (Middle price)
        group['hl2'] = (group['high'] + group['low']) / 2

        # Calculate Upper and Lower Bands
        group['upperband'] = group['hl2'] + (factor * group['atr'])
        group['lowerband'] = group['hl2'] - (factor * group['atr'])

        # Initialize Supertrend arrays

        
        supertrend = [-1] * len(group)
        supertrend_value = [None] * len(group)

        # Calculate Supertrend
        for i in range(1, len(group)):
            if group['close'].iloc[i] > group['upperband'].iloc[i - 1]:  # Bullish signal
                supertrend[i] = 1
                supertrend_value[i] = group['lowerband'].iloc[i]
            elif group['close'].iloc[i] < group['lowerband'].iloc[i - 1]:  # Bearish signal
                supertrend[i] = -1
                supertrend_value[i] = group['upperband'].iloc[i]
            else:
                supertrend[i] = supertrend[i - 1]
                supertrend_value[i] = supertrend_value[i - 1]

        # Assign results to the DataFrame
        group['supertrend'] = supertrend 
        
        return group

    # Apply function per stock (tk), ensuring previous values are used
    df = df.groupby('tk', group_keys=False).apply(compute_supertrend)

    # Keep only relevant columns
    return df

# Apply to your DataFrame
df_resampled = calculate_supertrend_5min(df_resampled)

# Display results
df_resampled.to_csv('/Users/mahesh/Downloads/df_resampled.csv')

df_resampled


66


,tk,timestamp,open,close,high,low,volume,previous_close,tr1,tr2,tr3,tr,atr,hl2,upperband,lowerband,supertrend
0,8479,2025-03-12 09:15:00,2290.25,2268.50,2293.95,2268.45,566887.0,NaN,25.50,NaN,NaN,25.50,25.50,2281.200,2306.700,2255.700,-1
1,8479,2025-03-12 09:20:00,2269.25,2264.85,2269.30,2264.50,314082.0,2268.50,4.80,0.80,4.00,4.80,4.80,2266.900,2271.700,2262.100,-1
2,8479,2025-03-12 09:25:00,2264.45,2258.35,2266.55,2256.00,2358274.0,2264.85,10.55,1.70,8.85,10.55,10.55,2261.275,2271.825,2250.725,-1
3,8479,2025-03-12 09:30:00,2257.65,2262.95,2263.90,2255.10,694079.0,2258.35,8.80,5.55,3.25,8.80,8.80,2259.500,2268.300,2250.700,-1
4,8479,2025-03-12 09:35:00,2264.40,2267.50,2269.30,2260.75,657132.0,2262.95,8.55,6.35,2.20,8.55,8.55,2265.025,2273.575,2256.475,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,8479,2025-03-12 14:20:00,2258.00,2258.25,2260.20,2256.00,12263805.0,2257.95,4.20,2.25,1.95,4.20,4.20,2258.100,2262.300,2253.900,1
62,8479,2025-03-12 14:25:00,2257.75,2257.55,2261.00,2256.95,14398791.0,2258.25,4.05,2.75,1.30,4.05,4.05,2258.975,2263.025,2254.925,1
63,8479,2025-03-12 14:30:00,2257.35,2259.65,2261.00,2256.10,16346838.0,2257.55,4.90,3.45,1.45,4.90,4.90,2258.550,2263.450,2253.650,1
64,8479,2025-03-12 14:35:00,2260.85,2261.80,2262.50,2259.50,14705028.0,2259.65,3.00,2.85,0.15,3.00,3.00,2261.000,2264.000,2258.000,1


In [ ]:
def atr(DF,n):
    "function to calculate True Range and Average True Range"
    df = DF.copy()
    df['H-L']=abs(df['high']-df['low'])
    df['H-PC']=abs(df['high']-df['close'].shift(1))
    df['L-PC']=abs(df['low']-df['close'].shift(1))
    df['TR']=df[['H-L','H-PC','L-PC']].max(axis=1,skipna=False)
    df['ATR'] = df['TR'].ewm(com=n,min_periods=n).mean()
    return df['ATR']

def supertrend(DF,n,m):
    """function to calculate Supertrend given historical candle data
        n = n day ATR - usually 7 day ATR is used
        m = multiplier - usually 2 or 3 is used"""
    df = DF.copy()
    df['ATR'] = atr(df,n)
    df["B-U"]=((df['high']+df['low'])/2) + m*df['ATR'] 
    df["B-L"]=((df['high']+df['low'])/2) - m*df['ATR']
    df["U-B"]=df["B-U"]
    df["L-B"]=df["B-L"]
    ind = df.index
    for i in range(n,len(df)):
        if df['close'][i-1]<=df['U-B'][i-1]:
            df.loc[ind[i],'U-B']=min(df['B-U'][i],df['U-B'][i-1])
        else:
            df.loc[ind[i],'U-B']=df['B-U'][i]    
    for i in range(n,len(df)):
        if df['close'][i-1]>=df['L-B'][i-1]:
            df.loc[ind[i],'L-B']=max(df['B-L'][i],df['L-B'][i-1])
        else:
            df.loc[ind[i],'L-B']=df['B-L'][i]  
    df['Strend']=np.nan
    for test in range(n,len(df)):
        if df['close'][test-1]<=df['U-B'][test-1] and df['close'][test]>df['U-B'][test]:
            df.loc[ind[test],'Strend']=df['L-B'][test]
            break
        if df['close'][test-1]>=df['L-B'][test-1] and df['close'][test]<df['L-B'][test]:
            df.loc[ind[test],'Strend']=df['U-B'][test]
            break
    for i in range(test+1,len(df)):
        if df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]<=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
        elif  df['Strend'][i-1]==df['U-B'][i-1] and df['close'][i]>=df['U-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]>=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['L-B'][i]
        elif df['Strend'][i-1]==df['L-B'][i-1] and df['close'][i]<=df['L-B'][i]:
            df.loc[ind[i],'Strend']=df['U-B'][i]
    return df['Strend']